In [2]:
"""
Extrae la serie de temperatura quincenal (media, desviación estándar y
varianza TEMPORAL, es decir entre los días dentro de cada quincena -
no confundir con el std ESPACIAL que calculamos en terrain_profile.py)
para un punto, usando ERA5-Land Daily Aggregates.

CÓMO LEER EL RESULTADO
-----------------------
media_C   Temperatura media de esa quincena, en °C.
std_C     Cuánto varió la temperatura día a día dentro de esa quincena.
          Alto = quincena con días muy dispares (ej. mezcla de días
          frescos y calurosos); bajo = temperatura estable en el período.
var_C     La misma variabilidad, en varianza (std al cuadrado).
"""
import calendar
from datetime import date, datetime, timedelta
from pathlib import Path
import pandas as pd
import ee
ee.Initialize()


def build_biweekly_periods(start_date, end_date):
    """
    Genera la lista de periodos quincenales (1-15, 16-fin de mes) entre
    start_date y end_date, en Python puro (sin llamadas a GEE todavía).
    Solo incluye quincenas ya completadas (period_end <= end_date + 1 día),
    para no meter quincenas parciales/futuras con datos incompletos.
    """
    periods = []
    current = date(start_date.year, start_date.month, 1)

    while current <= end_date:
        year, month = current.year, current.month

        q1_start = date(year, month, 1)
        q1_end = date(year, month, 16)  # exclusivo en filterDate -> cubre días 1-15

        q2_start = date(year, month, 16)
        if month == 12:
            q2_end = date(year + 1, 1, 1)
        else:
            q2_end = date(year, month + 1, 1)

        # Solo agregamos la quincena si ya terminó completamente
        if q1_end <= end_date + timedelta(days=1):
            periods.append((f'{year}-{month:02d}_Q1', q1_start, q1_end))
        if q2_end <= end_date + timedelta(days=1):
            periods.append((f'{year}-{month:02d}_Q2', q2_start, q2_end))

        current = date(year + 1, 1, 1) if month == 12 else date(year, month + 1, 1)

    return periods


def get_temperature_biweekly(lat, lon, start_date="2016-01-01", end_date=None):
    """
    lat, lon: coordenadas del punto
    start_date: fecha de inicio (str "YYYY-MM-DD")
    end_date: fecha de fin (str "YYYY-MM-DD"); None = hoy.
              OJO: ERA5-Land tiene algunos días de latencia en su
              publicación -> las quincenas más recientes podrían no
              estar disponibles aún aunque "ya pasaron" en el calendario.
    """
    start = datetime.strptime(start_date, "%Y-%m-%d").date()
    end = datetime.strptime(end_date, "%Y-%m-%d").date() if end_date else date.today()

    periods = build_biweekly_periods(start, end)
    print(f"[DEBUG] {len(periods)} quincenas a procesar, desde {start} hasta {end}")

    point = ee.Geometry.Point([lon, lat])

    era5_coll = (
        ee.ImageCollection('ECMWF/ERA5_LAND/DAILY_AGGR')
        .select('temperature_2m')
        .filterDate(str(start), str(end + timedelta(days=1)))
    )

    combined_reducer = (
        ee.Reducer.mean()
        .combine(reducer2=ee.Reducer.stdDev(), sharedInputs=True)
        .combine(reducer2=ee.Reducer.variance(), sharedInputs=True)
    )

    # Armamos la lista de periodos como ee.List de diccionarios, para
    # procesarlos TODOS en el servidor con .map() -> una sola llamada
    # de red al final, en vez de una por quincena (240 antes).
    ee_periods = ee.List([
        {'label': label, 'start': str(p_start), 'end': str(p_end)}
        for label, p_start, p_end in periods
    ])

    def compute_period(period):
        period = ee.Dictionary(period)
        p_start = ee.Date(period.get('start'))
        p_end = ee.Date(period.get('end'))

        filtered = era5_coll.filterDate(p_start, p_end)
        stats = filtered.reduce(combined_reducer).reduceRegion(
            reducer=ee.Reducer.first(),
            geometry=point,
            scale=9000,
            maxPixels=1e9
        )

        return ee.Feature(
            None,
            stats
            .set('label', period.get('label'))
            .set('periodo_inicio', p_start.format('YYYY-MM-dd'))
            .set('periodo_fin', p_end.advance(-1, 'day').format('YYYY-MM-dd'))  # último día incluido
        )

    features = ee.FeatureCollection(ee_periods.map(compute_period))
    result = features.getInfo()  # única llamada de red para TODOS los periodos

    rows = []
    for f in result['features']:
        props = f['properties']
        mean_k = props.get('temperature_2m_mean')
        std_k = props.get('temperature_2m_stdDev')
        var_k = props.get('temperature_2m_variance')

        rows.append({
            'periodo_inicio': props.get('periodo_inicio'),
            'periodo_fin': props.get('periodo_fin'),
            'label': props.get('label'),
            # 'lat': lat,
            # 'lon': lon,
            'media_C': round(mean_k - 273.15, 2) if mean_k is not None else None,
            'std_C': round(std_k, 2) if std_k is not None else None,
            'var_C': round(var_k, 2) if var_k is not None else None,
        })

    df = pd.DataFrame(rows)
    df['periodo_inicio'] = pd.to_datetime(df['periodo_inicio'])
    df = df.sort_values('periodo_inicio').reset_index(drop=True)
    return df


def save_temperature_profile(df, out_prefix="temperature_profile", output_dir="../databases"):
    """
    Guarda la serie de temperatura con timestamp en el nombre:
    {out_prefix}-vYYMMDDHHMMSS.csv (mismo patrón que terrain_profile/soil_profile)
    """
    timestamp = datetime.now().strftime("%y%m%d%H%M%S")
    filename = f"{out_prefix}-v{timestamp}.csv"

    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    out_path = output_path / filename

    df.to_csv(out_path, index=False)
    print(f"CSV guardado en {out_path} ({df.shape[0]}x{df.shape[1]})")
    return out_path


if __name__ == "__main__":
    
    LAT = -19.689669877950884
    LON = 147.22717515914223
    
# El Playon         --||     7.4584221918243045,    -73.222052853104
# Finca Matanza     --||     7.300921,              -73.009794
# Sugarcane_COL     --||     3.580109040361371,     -76.31299479308868
# Sugarcane_QLD     --||     -19.689669877950884,   147.22717515914223

    df = get_temperature_biweekly(LAT, LON, start_date="2016-01-01")
    out_path = save_temperature_profile(df)
    print(df.head(10))

[DEBUG] 254 quincenas a procesar, desde 2016-01-01 hasta 2026-08-06
CSV guardado en ../databases/temperature_profile-v260806172638.csv (254x6)
  periodo_inicio periodo_fin       label  media_C  std_C  var_C
0     2016-01-01  2016-01-15  2016-01_Q1    28.03   1.30   1.69
1     2016-01-16  2016-01-31  2016-01_Q2    28.35   1.06   1.12
2     2016-02-01  2016-02-15  2016-02_Q1    28.11   1.01   1.02
3     2016-02-16  2016-02-29  2016-02_Q2    29.07   1.30   1.68
4     2016-03-01  2016-03-15  2016-03_Q1    26.32   1.23   1.52
5     2016-03-16  2016-03-31  2016-03_Q2    26.06   0.84   0.71
6     2016-04-01  2016-04-15  2016-04_Q1    25.46   0.61   0.37
7     2016-04-16  2016-04-30  2016-04_Q2    24.50   0.42   0.18
8     2016-05-01  2016-05-15  2016-05_Q1    24.42   1.32   1.75
9     2016-05-16  2016-05-31  2016-05_Q2    24.05   0.79   0.63
